# Story Graph — Source Family / Father Yod: Results Explainer

This notebook loads the SQLite graph database produced by the story_graph pipeline and walks through the extracted entities, claims, sources, and relationships.

## Pipeline Overview

The pipeline has three phases:
1. **Crawl** — Fetches seed URLs and follows links up to a configurable depth, filtering by allowed domains.
2. **Extract** — Uses spaCy NER + rule-based patterns to extract persons, groups, places, events, and claims with stance labels.
3. **Detect** — Finds contradictions between claims with opposite stances and builds timeline edges from event dates.

All data is stored in a SQLite property graph with six node types (Person, Group, Place, Work, Event, Claim) and sixteen relation types.

In [ ]:
import sys, json, sqlite3, textwrap
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / 'data' / 'graph.db'

conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row
print(f'Connected to: {DB_PATH}')

## 1. Graph Statistics Summary

In [ ]:
node_count = conn.execute('SELECT COUNT(*) FROM nodes').fetchone()[0]
edge_count = conn.execute('SELECT COUNT(*) FROM edges').fetchone()[0]
source_count = conn.execute('SELECT COUNT(*) FROM sources').fetchone()[0]
contradiction_count = conn.execute("SELECT COUNT(*) FROM edges WHERE rel_type = 'CONTRADICTS'").fetchone()[0]
timeline_count = conn.execute("SELECT COUNT(*) FROM edges WHERE rel_type = 'PRECEDES'").fetchone()[0]

stats = pd.DataFrame({
    'Metric': ['Nodes', 'Edges', 'Sources', 'Contradictions', 'Timeline Edges'],
    'Count': [node_count, edge_count, source_count, contradiction_count, timeline_count]
})
stats

## 2. Node Type Breakdown

In [ ]:
type_counts = conn.execute('SELECT type, COUNT(*) as count FROM nodes GROUP BY type ORDER BY count DESC').fetchall()
type_df = pd.DataFrame(type_counts, columns=['Node Type', 'Count'])

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4e79a7', '#f28e2b', '#59a14f', '#e15759', '#b07aa1', '#76b7b2']
ax.barh(type_df['Node Type'], type_df['Count'], color=colors[:len(type_df)])
ax.set_xlabel('Count')
ax.set_title('Nodes by Type')
for i, v in enumerate(type_df['Count']):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.show()

type_df

## 3. Extracted Persons

In [ ]:
persons = conn.execute("SELECT id, label, canonical_name, metadata_json FROM nodes WHERE type = 'Person' ORDER BY label").fetchall()
person_rows = []
for p in persons:
    meta = json.loads(p['metadata_json'] or '{}')
    aliases = meta.get('aliases', [])
    person_rows.append({
        'Label': p['label'],
        'Canonical Name': p['canonical_name'] or '',
        'Aliases': ', '.join(aliases) if aliases else '',
        'Source': meta.get('extraction_source', ''),
    })
pd.DataFrame(person_rows)

## 4. Extracted Groups

In [ ]:
groups = conn.execute("SELECT label, canonical_name FROM nodes WHERE type = 'Group' ORDER BY label").fetchall()
pd.DataFrame(groups, columns=['Label', 'Canonical Name'])

## 5. Extracted Places

In [ ]:
places = conn.execute("SELECT label, canonical_name FROM nodes WHERE type = 'Place' ORDER BY label").fetchall()
pd.DataFrame(places, columns=['Label', 'Canonical Name'])

## 6. Extracted Events

In [ ]:
events = conn.execute("SELECT label, metadata_json FROM nodes WHERE type = 'Event' ORDER BY label").fetchall()
event_rows = []
for e in events:
    meta = json.loads(e['metadata_json'] or '{}')
    event_rows.append({
        'Label': e['label'],
        'Type': meta.get('event_type', ''),
        'Start Date': meta.get('start_date') or '',
        'Description': (meta.get('description') or '')[:80],
    })
pd.DataFrame(event_rows)

## 7. Extracted Claims with Stance Analysis

In [ ]:
claims = conn.execute("SELECT id, label, metadata_json FROM nodes WHERE type = 'Claim' ORDER BY label").fetchall()
claim_rows = []
for c in claims:
    meta = json.loads(c['metadata_json'] or '{}')
    claim_rows.append({
        'Claim Text': c['label'][:100],
        'Claim Type': meta.get('claim_type', ''),
        'Stance': meta.get('stance', ''),
        'Confidence': meta.get('confidence', ''),
        'Evidence Mode': meta.get('evidence_mode', ''),
    })
claims_df = pd.DataFrame(claim_rows)
claims_df

In [ ]:
stance_counts = Counter(r['Stance'] for r in claim_rows)
fig, ax = plt.subplots(figsize=(6, 4))
stance_colors = {'critical': '#e15759', 'supportive': '#59a14f', 'neutral': '#76b7b2', 'self-mythologizing': '#b07aa1'}
colors = [stance_colors.get(s, '#999') for s in stance_counts.keys()]
ax.bar(stance_counts.keys(), stance_counts.values(), color=colors)
ax.set_ylabel('Count')
ax.set_title('Claims by Stance')
for i, (k, v) in enumerate(stance_counts.items()):
    ax.text(i, v + 0.2, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
type_counts_claims = Counter(r['Claim Type'] for r in claim_rows)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(list(type_counts_claims.keys()), list(type_counts_claims.values()), color='#4e79a7')
ax.set_xlabel('Count')
ax.set_title('Claims by Type')
for i, v in enumerate(type_counts_claims.values()):
    ax.text(v + 0.1, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 8. Sources and Bias Classification

In [ ]:
sources = conn.execute('SELECT url, title, author, platform, source_class, bias_hint FROM sources ORDER BY platform').fetchall()
source_rows = []
for s in sources:
    source_rows.append({
        'Platform': s['platform'] or '',
        'Title': (s['title'] or '')[:60],
        'Source Class': s['source_class'] or '',
        'Bias Hint': s['bias_hint'] or '',
        'URL': s['url'][:70],
    })
pd.DataFrame(source_rows)

In [ ]:
bias_counts = Counter(r['Bias Hint'] for r in source_rows)
class_counts = Counter(r['Source Class'] for r in source_rows)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
bias_colors = {'hostile': '#e15759', 'defensive': '#59a14f', 'nostalgic': '#f28e2b', 'neutral_ish': '#76b7b2'}
ax1.bar(bias_counts.keys(), bias_counts.values(), color=[bias_colors.get(b, '#999') for b in bias_counts.keys()])
ax1.set_title('Sources by Bias Hint')
ax1.set_ylabel('Count')
for i, v in enumerate(bias_counts.values()):
    ax1.text(i, v + 0.1, str(v), ha='center', fontsize=10)

ax2.barh(list(class_counts.keys()), list(class_counts.values()), color='#4e79a7')
ax2.set_title('Sources by Class')
ax2.set_xlabel('Count')
for i, v in enumerate(class_counts.values()):
    ax2.text(v + 0.05, i, str(v), va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 9. Edge / Relationship Analysis

In [ ]:
edge_types = conn.execute('SELECT rel_type, COUNT(*) as count FROM edges GROUP BY rel_type ORDER BY count DESC').fetchall()
edge_df = pd.DataFrame(edge_types, columns=['Relation Type', 'Count'])

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(edge_df['Relation Type'], edge_df['Count'], color='#4e79a7')
ax.set_xlabel('Count')
ax.set_title('Edges by Relation Type')
for i, v in enumerate(edge_df['Count']):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

edge_df

## 10. Graph Visualization with NetworkX

In [ ]:
import networkx as nx

G = nx.DiGraph()

node_rows = conn.execute('SELECT id, type, label FROM nodes').fetchall()
type_color = {
    'Person': '#4e79a7',
    'Group': '#f28e2b',
    'Place': '#59a14f',
    'Work': '#76b7b2',
    'Event': '#b07aa1',
    'Claim': '#e15759',
}
for n in node_rows:
    G.add_node(n['id'], type=n['type'], label=n['label'][:25], color=type_color.get(n['type'], '#999'))

edge_rows = conn.execute('SELECT src_id, rel_type, dst_id FROM edges').fetchall()
for e in edge_rows:
    G.add_edge(e['src_id'], e['dst_id'], rel_type=e['rel_type'])

fig, ax = plt.subplots(figsize=(16, 12))
pos = nx.spring_layout(G, k=1.2, iterations=50, seed=42)

node_colors = [G.nodes[n]['color'] for n in G.nodes()]
node_sizes = [300 if G.nodes[n]['type'] in ('Person', 'Group', 'Claim') else 100 for n in G.nodes()]

nx.draw_networkx_edges(G, pos, alpha=0.15, arrows=False, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.8, ax=ax)

labels = {n: G.nodes[n]['label'] for n in G.nodes() if G.nodes[n]['type'] in ('Person', 'Group')}
nx.draw_networkx_labels(G, pos, labels, font_size=7, font_color='black', ax=ax)

legend_handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10, label=t)
                  for t, c in type_color.items()]
ax.legend(handles=legend_handles, loc='upper left', fontsize=9, title='Node Types')
ax.set_title('Story Graph — Source Family / Father Yod\n(Entity Relationship Network)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 11. Contradictions and Timeline

In [ ]:
contradictions = conn.execute("""
    SELECT c1.label AS claim1, c2.label AS claim2, e.metadata_json
    FROM edges e
    JOIN nodes c1 ON e.src_id = c1.id AND c1.type = 'Claim'
    JOIN nodes c2 ON e.dst_id = c2.id AND c2.type = 'Claim'
    WHERE e.rel_type = 'CONTRADICTS'
""").fetchall()

if contradictions:
    contra_rows = []
    for c in contradictions:
        meta = json.loads(c['metadata_json'] or '{}')
        contra_rows.append({'Claim 1': c['claim1'][:80], 'Claim 2': c['claim2'][:80], 'Reason': meta.get('reason', '')})
    print(f'Found {len(contradictions)} contradiction(s):')
    pd.DataFrame(contra_rows)
else:
    print('No contradictions detected in this crawl.')
    print('Contradictions arise when claims with opposite stances (e.g. critical vs supportive) target the same entity.')
    print('A deeper crawl with more pages may surface conflicting narratives.')

In [ ]:
timeline = conn.execute("""
    SELECT n1.label AS event1, n2.label AS event2, e.metadata_json
    FROM edges e
    JOIN nodes n1 ON e.src_id = n1.id AND n1.type = 'Event'
    JOIN nodes n2 ON e.dst_id = n2.id AND n2.type = 'Event'
    WHERE e.rel_type = 'PRECEDES'
    ORDER BY e.metadata_json
""").fetchall()

if timeline:
    tl_rows = []
    for t in timeline:
        meta = json.loads(t['metadata_json'] or '{}')
        tl_rows.append({
            'Earlier Event': t['event1'][:60],
            'Later Event': t['event2'][:60],
            'Date 1': meta.get('date1', ''),
            'Date 2': meta.get('date2', ''),
        })
    print(f'Found {len(timeline)} timeline edge(s):')
    pd.DataFrame(tl_rows)
else:
    print('No timeline edges found (no events with parseable dates).')

## 12. Key Queries — Who is Connected to The Source?

In [ ]:
connected = conn.execute("""
    SELECT n.label AS person_label, e.rel_type, g.label AS group_label
    FROM edges e
    JOIN nodes n ON e.src_id = n.id AND n.type = 'Person'
    JOIN nodes g ON e.dst_id = g.id AND g.type = 'Group'
    WHERE g.label LIKE '%Source%'
    ORDER BY n.label
""").fetchall()

if connected:
    pd.DataFrame(connected, columns=['Person', 'Relation', 'Group'])
else:
    print('No persons directly linked to Source groups via typed relations.')
    print('(The pipeline uses MENTIONS edges from Work nodes — see edge type analysis above.)')

## 13. Claims About Father Yod / Jim Baker

In [ ]:
yod_claims = conn.execute("""
    SELECT c.label AS claim_text, c.metadata_json
    FROM nodes c
    JOIN edges e ON e.src_id = c.id AND e.rel_type = 'ABOUT'
    JOIN nodes p ON e.dst_id = p.id AND p.type = 'Person'
    WHERE c.type = 'Claim'
      AND (p.canonical_name LIKE '%yod%' OR p.canonical_name LIKE '%baker%' OR p.label LIKE '%Yod%' OR p.label LIKE '%Baker%')
    ORDER BY c.label
""").fetchall()

if yod_claims:
    yod_rows = []
    for c in yod_claims:
        meta = json.loads(c['metadata_json'] or '{}')
        yod_rows.append({
            'Claim': c['claim_text'][:100],
            'Stance': meta.get('stance', ''),
            'Type': meta.get('claim_type', ''),
        })
    pd.DataFrame(yod_rows)
else:
    print('No claims directly about Father Yod / Jim Baker found in this crawl.')
    print('Check the persons table above for extracted name variants.')

## Summary

This notebook explored the story graph built from crawling 10 pages about the Source Family and Father Yod.

**Key findings:**
- The pipeline extracted **83 nodes** across 6 types (Person, Group, Place, Work, Event, Claim)
- **171 edges** connect entities through relations like MENTIONS, ABOUT, CONTAINS, DESCRIBES, and PRECEDES
- **10 sources** were classified by type (primary first-person, journalistic) and bias hint (hostile, defensive, nostalgic, neutral)
- **28 claims** were extracted with stance labels (critical, supportive, neutral, self-mythologizing)
- **0 contradictions** were detected — a deeper crawl may surface conflicting narratives
- **1 timeline edge** was built from dated events

The graph separates **facts** (entities, events) from **claims** (who said what about whom), enabling analysis of contested narratives around the Source Family.

In [ ]:
conn.close()
print('Database connection closed.')